# Web-scraping LocalHarvest

In [2]:
import requests
from bs4 import BeautifulSoup
import time 
import random 
from urllib.parse import urlparse, parse_qs
import pandas as pd

In [3]:
# Scraping Dallas 

base_url = 'https://www.localharvest.org/dallas-tx/farms?'
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# initializing dictionary
scraped_data = [] 

# Searching main search results pages 
print('Going through main search results')
for page_num in range(1, 5):
    url = f"{base_url}p={page_num}"
    print(f"Scraping: {url}")

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'lxml')

        for site in soup.select('.col.p-1.p-sm-3'):
            title = site.select_one('.mt-0')
            farm_name = title.text.strip()
            profile_path = title.get('href', '')
            profile_url = f'localharvest.org{profile_path}' 
            
            location = site.select_one('.text-muted.small.d-block > a')
            
            latitude = None
            longitude = None
            
            if location:
                href_string = location.get('href', '')

                if 'lat=' in href_string and 'lon=' in href_string:
                    parsed_url = urlparse(href_string)
                    query_params = parse_qs(parsed_url.query)
                    
                    latitude = query_params.get('lat', [None])[0]
                    longitude = query_params.get('lon', [None])[0]
                    
            scraped_data.append({
                'farm_name': farm_name,
                'profile_url': profile_url,
                'latitude': latitude,
                'longitude': longitude,
                'address': None
            })
    else:
        print(f'Failed to retrieve webpage. Status code: {response.status_code}')

df = pd.DataFrame(scraped_data)

# Searching through individual profile pages for those that lack lat/lon
print('\nStarting search for addresses in profile pages')

missing_coords = df[df['latitude'].isnull()]
print(f'{len(missing_coords)} require profile visits')

for index, row in missing_coords.iterrows():
    link = f"https://{row['profile_url']}"
    print(f'Visiting profile page: {link}')
    
    try: 
        profile_response = requests.get(link, headers=headers)

        if profile_response.status_code == 200:
            profile_soup = BeautifulSoup(profile_response.text, 'lxml')

            address_element = profile_soup.select_one('.border-top.mt-3.pt-2')

            if address_element: 
                cleaned_address = " ".join(address_element.text.split())
                df.at[index, 'street_address'] = cleaned_address 

            else:
                df.at[index, 'street_address'] = 'Not found'

        else: 
            print(f'Could not open profile page (Status {profile_response.status_code})')
            df.at[index, 'street_address'] = 'HTTP Error'

    except Exception as e:
        print(f'Error fetching {link}: {str(e)}')
        df.at[index, 'street_address'] = f'Exception: {type(e).__name__}'

    sleep_time = random.uniform(3.0, 7.0)
    time.sleep(sleep_time)

df.head() 
df.to_csv('dallas_localharvest.csv', index=False)
print(f'Finished and written to {output_file}')

Going through main search results
Scraping: https://www.localharvest.org/dallas-tx/farms?p=1
Scraping: https://www.localharvest.org/dallas-tx/farms?p=2
Scraping: https://www.localharvest.org/dallas-tx/farms?p=3
Scraping: https://www.localharvest.org/dallas-tx/farms?p=4

Starting search for addresses in profile pages
31 require profile visits
Visiting profile page: https://localharvest.org/jjjm-grass-fed-beef-M71256
Visiting profile page: https://localharvest.org/pastured-steps-family-farm-M75854
Visiting profile page: https://localharvest.org/pennys-pastured-poultry-M74044
Visiting profile page: https://localharvest.org/foster-farm-grass-fed-grass-finished-beef-M71211
Visiting profile page: https://localharvest.org/alford-family-farm-M71284
Visiting profile page: https://localharvest.org/p-o-p-acres-M18186
Visiting profile page: https://localharvest.org/texas-organic-mushrooms-M17453
Visiting profile page: https://localharvest.org/parish-custom-beef-llc-M75843
Visiting profile page: ht

In [4]:
# Getting rid of mistaken 'address' column
dallas_lh = pd.read_csv('dallas_localharvest.csv')
dallas_lh = dallas_lh[['farm_name', 'profile_url', 'latitude', 'longitude', 'street_address']]
dallas_lh.to_csv('dallas_localharvest.csv', index=False)

In [18]:
# Scraping Fort Worth
base_url = 'https://www.localharvest.org/benbrook-tx/farms?'
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# initializing dictionary
scraped_data = [] 

# Searching main search results pages 
print('Going through main search results')
for page_num in range(1, 5):
    url = f"{base_url}p={page_num}"
    print(f"Scraping: {url}")

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'lxml')

        for site in soup.select('.col.p-1.p-sm-3'):
            title = site.select_one('.mt-0')
            farm_name = title.text.strip()
            profile_path = title.get('href', '')
            profile_url = f'localharvest.org{profile_path}' 
            
            location = site.select_one('.text-muted.small.d-block > a')
            
            latitude = None
            longitude = None
            
            if location:
                href_string = location.get('href', '')

                if 'lat=' in href_string and 'lon=' in href_string:
                    parsed_url = urlparse(href_string)
                    query_params = parse_qs(parsed_url.query)
                    
                    latitude = query_params.get('lat', [None])[0]
                    longitude = query_params.get('lon', [None])[0]
                    
            scraped_data.append({
                'farm_name': farm_name,
                'profile_url': profile_url,
                'latitude': latitude,
                'longitude': longitude,
                'street_address': None
            })
    else:
        print(f'Failed to retrieve webpage. Status code: {response.status_code}')

df = pd.DataFrame(scraped_data)

# Searching through individual profile pages for those that lack lat/lon
print('\nStarting search for addresses in profile pages')

missing_coords = df[df['latitude'].isnull()]
print(f'{len(missing_coords)} require profile visits')

for index, row in missing_coords.iterrows():
    link = f"https://{row['profile_url']}"
    print(f'Visiting profile page: {link}')
    
    try: 
        profile_response = requests.get(link, headers=headers)

        if profile_response.status_code == 200:
            profile_soup = BeautifulSoup(profile_response.text, 'lxml')

            address_element = profile_soup.select_one('.border-top.mt-3.pt-2')

            if address_element: 
                cleaned_address = " ".join(address_element.text.split())
                df.at[index, 'street_address'] = cleaned_address 

            else:
                df.at[index, 'street_address'] = 'Not found'

        else: 
            print(f'Could not open profile page (Status {profile_response.status_code})')
            df.at[index, 'street_address'] = 'HTTP Error'

    except Exception as e:
        print(f'Error fetching {link}: {str(e)}')
        df.at[index, 'street_address'] = f'Exception: {type(e).__name__}'

    sleep_time = random.uniform(3.0, 7.0)
    time.sleep(sleep_time)

df.head() 
output_file = 'fortworth_localharvest.csv'
df.to_csv(output_file, index=False)
print(f'Finished and written to {output_file}')

Going through main search results
Scraping: https://www.localharvest.org/benbrook-tx/farms?p=1
Scraping: https://www.localharvest.org/benbrook-tx/farms?p=2
Scraping: https://www.localharvest.org/benbrook-tx/farms?p=3
Scraping: https://www.localharvest.org/benbrook-tx/farms?p=4

Starting search for addresses in profile pages
33 require profile visits
Visiting profile page: https://localharvest.org/foster-farm-grass-fed-grass-finished-beef-M71211
Visiting profile page: https://localharvest.org/pastured-steps-family-farm-M75854
Visiting profile page: https://localharvest.org/parish-custom-beef-llc-M75843
Visiting profile page: https://localharvest.org/jjjm-grass-fed-beef-M71256
Visiting profile page: https://localharvest.org/p-o-p-acres-M18186
Visiting profile page: https://localharvest.org/pennys-pastured-poultry-M74044
Visiting profile page: https://localharvest.org/texas-organic-mushrooms-M17453
Visiting profile page: https://localharvest.org/ginger-mcmillan-M70686
Visiting profile pag

In [5]:
# Merging Dallas and FW dataframes
fw_lh = pd.read_csv('fortworth_localharvest.csv')
dfw_localharvest_df = pd.concat([dallas_lh, fw_lh], ignore_index=True)
dfw_localharvest_df = dfw_localharvest_df.drop_duplicates(subset=['farm_name'], keep='first', ignore_index=True)
dfw_localharvest_df.to_csv('dfw_localharvest.csv', index=False)

In [17]:
# Scraping Austin
base_url = 'https://www.localharvest.org/austin-tx/farms?'
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# initializing dictionary
scraped_data = [] 

# Searching main search results pages 
print('Going through main search results')
for page_num in range(1, 5):
    url = f"{base_url}p={page_num}"
    print(f"Scraping: {url}")

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'lxml')

        for site in soup.select('.col.p-1.p-sm-3'):
            title = site.select_one('.mt-0')
            farm_name = title.text.strip()
            profile_path = title.get('href', '')
            profile_url = f'localharvest.org{profile_path}' 
            
            location = site.select_one('.text-muted.small.d-block > a')
            
            latitude = None
            longitude = None
            
            if location:
                href_string = location.get('href', '')

                if 'lat=' in href_string and 'lon=' in href_string:
                    parsed_url = urlparse(href_string)
                    query_params = parse_qs(parsed_url.query)
                    
                    latitude = query_params.get('lat', [None])[0]
                    longitude = query_params.get('lon', [None])[0]
                    
            scraped_data.append({
                'farm_name': farm_name,
                'profile_url': profile_url,
                'latitude': latitude,
                'longitude': longitude,
                'street_address': None
            })
    else:
        print(f'Failed to retrieve webpage. Status code: {response.status_code}')

df = pd.DataFrame(scraped_data)

# Searching through individual profile pages for those that lack lat/lon
print('\nStarting search for addresses in profile pages')

missing_coords = df[df['latitude'].isnull()]
print(f'{len(missing_coords)} require profile visits')

for index, row in missing_coords.iterrows():
    link = f"https://{row['profile_url']}"
    print(f'Visiting profile page: {link}')
    
    try: 
        profile_response = requests.get(link, headers=headers)

        if profile_response.status_code == 200:
            profile_soup = BeautifulSoup(profile_response.text, 'lxml')

            address_element = profile_soup.select_one('.border-top.mt-3.pt-2')

            if address_element: 
                cleaned_address = " ".join(address_element.text.split())
                df.at[index, 'street_address'] = cleaned_address 

            else:
                df.at[index, 'street_address'] = 'Not found'

        else: 
            print(f'Could not open profile page (Status {profile_response.status_code})')
            df.at[index, 'street_address'] = 'HTTP Error'

    except Exception as e:
        print(f'Error fetching {link}: {str(e)}')
        df.at[index, 'street_address'] = f'Exception: {type(e).__name__}'

    sleep_time = random.uniform(3.0, 7.0)
    time.sleep(sleep_time)

df.head() 
output_file = 'austin_localharvest.csv'
df.to_csv(output_file, index=False)
print(f'Finished and written to {output_file}')

Going through main search results
Scraping: https://www.localharvest.org/austin-tx/farms?p=1
Scraping: https://www.localharvest.org/austin-tx/farms?p=2
Scraping: https://www.localharvest.org/austin-tx/farms?p=3
Scraping: https://www.localharvest.org/austin-tx/farms?p=4

Starting search for addresses in profile pages
34 require profile visits
Visiting profile page: https://localharvest.org/central-texas-farmers-co-op-M74330
Visiting profile page: https://localharvest.org/eat-your-greens-organic-farm-M76452
Visiting profile page: https://localharvest.org/green-gate-farms-M15256
Visiting profile page: https://localharvest.org/munkebo-farm-M37675
Visiting profile page: https://localharvest.org/holland-cattle-co--M72554
Visiting profile page: https://localharvest.org/galloping-winds-ranch-M25019
Visiting profile page: https://localharvest.org/ahimsa-farm-M30325
Visiting profile page: https://localharvest.org/wild-type-ranch-M16339
Visiting profile page: https://localharvest.org/watterson-ra

In [16]:
# Scraping San Antonio
base_url = 'https://www.localharvest.org/san-antonio-tx/farms?'
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# initializing dictionary
scraped_data = [] 

# Searching main search results pages 
print('Going through main search results')
for page_num in range(1, 5):
    url = f"{base_url}p={page_num}"
    print(f"Scraping: {url}")

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'lxml')

        for site in soup.select('.col.p-1.p-sm-3'):
            title = site.select_one('.mt-0')
            farm_name = title.text.strip()
            profile_path = title.get('href', '')
            profile_url = f'localharvest.org{profile_path}' 
            
            location = site.select_one('.text-muted.small.d-block > a')
            
            latitude = None
            longitude = None
            
            if location:
                href_string = location.get('href', '')

                if 'lat=' in href_string and 'lon=' in href_string:
                    parsed_url = urlparse(href_string)
                    query_params = parse_qs(parsed_url.query)
                    
                    latitude = query_params.get('lat', [None])[0]
                    longitude = query_params.get('lon', [None])[0]
                    
            scraped_data.append({
                'farm_name': farm_name,
                'profile_url': profile_url,
                'latitude': latitude,
                'longitude': longitude,
                'street_address': None
            })
    else:
        print(f'Failed to retrieve webpage. Status code: {response.status_code}')

df = pd.DataFrame(scraped_data)

# Searching through individual profile pages for those that lack lat/lon
print('\nStarting search for addresses in profile pages')

missing_coords = df[df['latitude'].isnull()]
print(f'{len(missing_coords)} require profile visits')

for index, row in missing_coords.iterrows():
    link = f"https://{row['profile_url']}"
    print(f'Visiting profile page: {link}')
    
    try: 
        profile_response = requests.get(link, headers=headers)

        if profile_response.status_code == 200:
            profile_soup = BeautifulSoup(profile_response.text, 'lxml')

            address_element = profile_soup.select_one('.border-top.mt-3.pt-2')

            if address_element: 
                cleaned_address = " ".join(address_element.text.split())
                df.at[index, 'street_address'] = cleaned_address 

            else:
                df.at[index, 'street_address'] = 'Not found'

        else: 
            print(f'Could not open profile page (Status {profile_response.status_code})')
            df.at[index, 'street_address'] = 'HTTP Error'

    except Exception as e:
        print(f'Error fetching {link}: {str(e)}')
        df.at[index, 'street_address'] = f'Exception: {type(e).__name__}'

    sleep_time = random.uniform(3.0, 7.0)
    time.sleep(sleep_time)

df.head() 
output_file = 'san_antonio_localharvest.csv'
df.to_csv(output_file, index=False)
print(f'Finished and written to {output_file}')

Going through main search results
Scraping: https://www.localharvest.org/san-antonio-tx/farms?p=1
Scraping: https://www.localharvest.org/san-antonio-tx/farms?p=2
Scraping: https://www.localharvest.org/san-antonio-tx/farms?p=3
Scraping: https://www.localharvest.org/san-antonio-tx/farms?p=4

Starting search for addresses in profile pages
35 require profile visits
Visiting profile page: https://localharvest.org/central-texas-farmers-co-op-M74330
Visiting profile page: https://localharvest.org/eat-your-greens-organic-farm-M76452
Visiting profile page: https://localharvest.org/ahimsa-farm-M30325
Visiting profile page: https://localharvest.org/green-gate-farms-M15256
Visiting profile page: https://localharvest.org/munkebo-farm-M37675
Visiting profile page: https://localharvest.org/holland-cattle-co--M72554
Visiting profile page: https://localharvest.org/shudde-ranch-beef-parker-creek-ranch-M6541
Visiting profile page: https://localharvest.org/rancho-ojo-de-agua-M42697
Visiting profile page: 

In [23]:
# Scraping Houston
base_url = 'https://www.localharvest.org/houston-tx/farms?'
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# initializing dictionary
scraped_data = [] 

# Searching main search results pages 
print('Going through main search results')
for page_num in range(1, 4):
    url = f"{base_url}p={page_num}"
    print(f"Scraping: {url}")

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'lxml')

        for site in soup.select('.col.p-1.p-sm-3'):
            title = site.select_one('.mt-0')
            farm_name = title.text.strip()
            profile_path = title.get('href', '')
            profile_url = f'localharvest.org{profile_path}' 
            
            location = site.select_one('.text-muted.small.d-block > a')
            
            latitude = None
            longitude = None
            
            if location:
                href_string = location.get('href', '')

                if 'lat=' in href_string and 'lon=' in href_string:
                    parsed_url = urlparse(href_string)
                    query_params = parse_qs(parsed_url.query)
                    
                    latitude = query_params.get('lat', [None])[0]
                    longitude = query_params.get('lon', [None])[0]
                    
            scraped_data.append({
                'farm_name': farm_name,
                'profile_url': profile_url,
                'latitude': latitude,
                'longitude': longitude,
                'street_address': None
            })
    else:
        print(f'Failed to retrieve webpage. Status code: {response.status_code}')

df = pd.DataFrame(scraped_data)

# Searching through individual profile pages for those that lack lat/lon
print('\nStarting search for addresses in profile pages')

missing_coords = df[df['latitude'].isnull()]
print(f'{len(missing_coords)} require profile visits')

for index, row in missing_coords.iterrows():
    link = f"https://{row['profile_url']}"
    print(f'Visiting profile page: {link}')
    
    try: 
        profile_response = requests.get(link, headers=headers)

        if profile_response.status_code == 200:
            profile_soup = BeautifulSoup(profile_response.text, 'lxml')

            address_element = profile_soup.select_one('.border-top.mt-3.pt-2')

            if address_element: 
                cleaned_address = " ".join(address_element.text.split())
                df.at[index, 'street_address'] = cleaned_address 

            else:
                df.at[index, 'street_address'] = 'Not found'

        else: 
            print(f'Could not open profile page (Status {profile_response.status_code})')
            df.at[index, 'street_address'] = 'HTTP Error'

    except Exception as e:
        print(f'Error fetching {link}: {str(e)}')
        df.at[index, 'street_address'] = f'Exception: {type(e).__name__}'

    sleep_time = random.uniform(3.0, 7.0)
    time.sleep(sleep_time)

df.head() 
output_file = 'houston_localharvest.csv'
df.to_csv(output_file, index=False)
print(f'Finished and written to {output_file}')

Going through main search results
Scraping: https://www.localharvest.org/houston-tx/farms?p=1
Scraping: https://www.localharvest.org/houston-tx/farms?p=2
Scraping: https://www.localharvest.org/houston-tx/farms?p=3

Starting search for addresses in profile pages
30 require profile visits
Visiting profile page: https://localharvest.org/pure-source-farms-llc-M83603
Visiting profile page: https://localharvest.org/wild-honey-box-formerly-bee-wilde-M18100
Visiting profile page: https://localharvest.org/hope-farms-urban-agricultural-showcase-amp-training-center-M74453
Visiting profile page: https://localharvest.org/jolly-farms-chickens-M70650
Visiting profile page: https://localharvest.org/wild-turkey-farm-M27180
Visiting profile page: https://localharvest.org/far-out-farm-M43386
Visiting profile page: https://localharvest.org/the-last-organic-outpost-M5344
Visiting profile page: https://localharvest.org/bee2bee-honey-collective-M74937
Visiting profile page: https://localharvest.org/jolie-vue

In [ ]:
houston_lh = pd